# VidNotes AI - Video Summarizer & Notes Generator

Yeh notebook **VidNotes AI** project ke saare main codebase, configurations, and running instructions ko document karta hai.

## Project Overview
VidNotes ek full-stack AI tool hai jo uploaded videos ko process karke unka **English aur Hinglish** dono mein:
1. **Saransh / Executive Summary**
2. **Khaas Baatein / Key Highlights**
3. **Annotated Padhai Notes / Detailed Study Notes**

generate karta hai, jisme dynamic dark glassmorphic UI aur GSAP smooth animations shamil hain.

## 1. Backend Architecture & API Integration

Backend **Node.js (Express)** aur **Multer** use karke video uploads receive karta hai, and official `@google/genai` Node SDK use karke Gemini API call karta hai.

### Backend Dependencies (`backend/package.json`):

In [ ]:
// backend/package.json
{
  "name": "vidnotes-backend",
  "version": "1.0.0",
  "type": "module",
  "main": "server.js",
  "scripts": {
    "start": "node server.js",
    "dev": "nodemon server.js"
  },
  "dependencies": {
    "@google/genai": "^2.8.0",
    "cors": "^2.8.5",
    "dotenv": "^16.4.5",
    "express": "^4.19.2",
    "multer": "^1.4.5-lts.1"
  },
  "devDependencies": {
    "nodemon": "^3.1.0"
  }
}


### Environment Variables Config Template (`backend/.env`):
Gemini API access karne ke liye apni key isme configure karein.

In [ ]:
# backend/.env
PORT=3000
# Add your Google Gemini API Key here
GEMINI_API_KEY=YOUR_GEMINI_API_KEY


### Express Server Code (`backend/server.js`):
Multer uploads and Gemini 2.0 Ingestion/Polling endpoints code:

In [ ]:
// backend/server.js
import express from 'express';
import cors from 'cors';
import multer from 'multer';
import dotenv from 'dotenv';
import path from 'path';
import fs from 'fs';
import { fileURLToPath } from 'url';
import { GoogleGenAI } from '@google/genai';

dotenv.config();

const __filename = fileURLToPath(import.meta.url);
const __dirname = path.dirname(__filename);

const app = express();
const PORT = process.env.PORT || 3000;

// Middleware
app.use(cors());
app.use(express.json());
app.use(express.urlencoded({ extended: true }));

// Ensure uploads directory exists
const uploadsDir = path.join(__dirname, 'uploads');
if (!fs.existsSync(uploadsDir)) {
  fs.mkdirSync(uploadsDir, { recursive: true });
}

// Multer storage configuration
const storage = multer.diskStorage({
  destination: (req, file, cb) => {
    cb(null, uploadsDir);
  },
  filename: (req, file, cb) => {
    const uniqueSuffix = Date.now() + '-' + Math.round(Math.random() * 1e9);
    cb(null, file.fieldname + '-' + uniqueSuffix + path.extname(file.originalname));
  }
});

const upload = multer({
  storage: storage,
  limits: { fileSize: 200 * 1024 * 1024 }, // 200MB limit
  fileFilter: (req, file, cb) => {
    const filetypes = /mp4|webm|avi|mov|mkv/;
    const extname = filetypes.test(path.extname(file.originalname).toLowerCase());
    const mimetype = filetypes.test(file.mimetype);

    if (mimetype && extname) {
      return cb(null, true);
    } else {
      cb(new Error('Only video files (mp4, webm, avi, mov, mkv) are allowed.'));
    }
  }
});

// JSON Schema for Gemini structured output
const responseSchema = {
  type: "OBJECT",
  properties: {
    english: {
      type: "OBJECT",
      properties: {
        summary: { 
          type: "STRING", 
          description: "An extremely detailed, comprehensive summary of the video (at least 2-3 long paragraphs, 150-250 words) detailing everything discussed, the core context, the speaker's arguments, and conclusions." 
        },
        keyPoints: {
          type: "ARRAY",
          items: { type: "STRING" },
          description: "A list of 8-12 comprehensive key highlights and detailed insights from the video."
        },
        notes: {
          type: "ARRAY",
          items: { type: "STRING" },
          description: "Extremely detailed, exhaustive step-by-step structured study notes covering every single point, argument, topic, formula, tool, or code block mentioned in the video in high detail with temporal context/timestamps."
        }
      },
      required: ["summary", "keyPoints", "notes"]
    },
    hinglish: {
      type: "OBJECT",
      properties: {
        summary: { 
          type: "STRING", 
          description: "Ek lamba aur detail mein likha hua natural Hinglish (Latin script Hindi) summary (kam se kam 2-3 bade paragraphs, 150-250 words) jo video ki har ek baat aur mudde ko achhe se cover kare." 
        },
        keyPoints: {
          type: "ARRAY",
          items: { type: "STRING" },
          description: "Video ki 8-12 sabse jaruri aur khaas baatein details ke sath (Hinglish mein)."
        },
        notes: {
          type: "ARRAY",
          items: { type: "STRING" },
          description: "Ekdum detailed, point-by-point exhaustive study notes Hinglish mein, jo video ki har ek topic, timestamps, and details ko thoroughly cover karein (koi bhi point chhootna nahi chahiye)."
        }
      },
      required: ["summary", "keyPoints", "notes"]
    }
  },
  required: ["english", "hinglish"]
};

// Route: Upload & Summarize Video
app.post('/api/summarize', upload.single('video'), async (req, res) => {
  const file = req.file;
  if (!file) {
    return res.status(400).json({ error: 'No video file uploaded.' });
  }

  const apiKey = process.env.GEMINI_API_KEY;
  if (!apiKey) {
    // Clean up local file first
    if (fs.existsSync(file.path)) fs.unlinkSync(file.path);
    return res.status(500).json({ 
      error: 'Gemini API Key is not configured on the server. Please add GEMINI_API_KEY to your .env file.' 
    });
  }

  console.log(`Received file: ${file.originalname} (${(file.size / (1024 * 1024)).toFixed(2)} MB)`);

  try {
    // Initialize Google Gen AI client
    const ai = new GoogleGenAI({ apiKey });

    console.log("Uploading file to Google Gemini File API...");
    let uploadResult = await ai.files.upload({
      file: file.path,
      config: {
        mimeType: file.mimetype,
      }
    });

    console.log(`File uploaded. Name: ${uploadResult.name}`);

    // Wait for the video to be processed (status ACTIVE)
    let isProcessed = false;
    let attempts = 0;
    const maxAttempts = 60; // 5 minutes max (5s interval)

    while (!isProcessed && attempts < maxAttempts) {
      attempts++;
      console.log(`Polling video status (Attempt ${attempts}/${maxAttempts})...`);
      
      const fileStatus = await ai.files.get({ name: uploadResult.name });
      console.log(`Status: ${fileStatus.state}`);

      if (fileStatus.state === 'ACTIVE') {
        isProcessed = true;
        uploadResult = fileStatus; // Update metadata with final active file
      } else if (fileStatus.state === 'FAILED') {
        throw new Error('Google Gemini Video Ingestion failed.');
      } else {
        await new Promise(resolve => setTimeout(resolve, 5000));
      }
    }

    if (!isProcessed) {
      throw new Error('Video ingestion timed out. Please try with a smaller or shorter video.');
    }

    console.log("Video processing complete. Ingesting content with Gemini...");

    // Send generation request to Gemini
    const prompt = `
      You are an elite research assistant. Analyze the uploaded video file in extreme detail.
      Your output must be highly detailed, exhaustive, and comprehensive. Do NOT give high-level, short summaries or brief bullet points.
      
      Generate:
      1. An exhaustive Summary of the video content covering all chapters/topics discussed, context, problems raised, and solutions offered. Write at least 2-3 long, descriptive paragraphs (150-250 words) for the summary.
      2. A list of 8-12 detailed Key Highlights, detailing not just what was said but why it is important.
      3. A large set of extremely comprehensive step-by-step Structured Study Notes. Transcribe and summarize every single subtopic, concept, step, code, formula, advice, or fact mentioned in the video, with temporal reference/timestamps. Ensure absolutely no important details are omitted.
      
      Provide all of these in BOTH English and Hinglish as requested by the JSON schema.
      Ensure the Hinglish translation is natural, flowing, and highly descriptive (Latin script Hindi used in common messaging).
    `;

    const candidateModels = [
      'gemini-2.5-flash',
      'gemini-2.5-flash-lite',
      'gemini-3.1-flash-lite',
      'gemini-3.5-flash'
    ];

    let response = null;
    let finalModelUsed = null;
    let lastError = null;

    for (const modelName of candidateModels) {
      try {
        console.log(`Attempting generation with model: ${modelName}...`);
        response = await ai.models.generateContent({
          model: modelName,
          contents: [
            {
              fileData: {
                fileUri: uploadResult.uri,
                mimeType: uploadResult.mimeType
              }
            },
            { text: prompt }
          ],
          config: {
            responseMimeType: 'application/json',
            responseSchema: responseSchema
          }
        });
        finalModelUsed = modelName;
        console.log(`AI Generation successful with model: ${modelName}!`);
        break; // Stop loop on success
      } catch (err) {
        console.warn(`Model ${modelName} failed: ${err.message}`);
        lastError = err;
      }
    }

    if (!response) {
      throw new Error(`All Gemini models failed. Last error: ${lastError ? lastError.message : 'Unknown error'}`);
    }

    // Clean up Gemini File API storage (important to prevent quota leaks)
    try {
      await ai.files.delete({ name: uploadResult.name });
      console.log("Cleaned up Gemini cloud file storage.");
    } catch (cleanupErr) {
      console.warn("Could not clean up Gemini cloud file:", cleanupErr.message);
    }

    // Clean up local temp file
    if (fs.existsSync(file.path)) {
      fs.unlinkSync(file.path);
      console.log("Cleaned up local temporary video file.");
    }

    // Parse and return JSON
    const parsedData = JSON.parse(response.text);
    return res.json({
      success: true,
      videoName: file.originalname,
      modelUsed: finalModelUsed,
      results: parsedData
    });

  } catch (error) {
    console.error("Error processing video:", error);

    // Clean up local file on failure
    if (file && fs.existsSync(file.path)) {
      fs.unlinkSync(file.path);
    }

    return res.status(500).json({ 
      error: error.message || 'An error occurred during video analysis.' 
    });
  }
});

// Start Server
app.listen(PORT, () => {
  console.log(`VidNotes Server is running on http://localhost:${PORT}`);
});


## 2. Frontend User Interface (React + GSAP + Tailwind v4)

Frontend React (Vite template) par build kiya hai. Isme **AeroGlass Dark** theme applied hai.

### Tailwind Configuration (`frontend/tailwind.config.js`):

In [ ]:
// frontend/tailwind.config.js
/** @type {import('tailwindcss').Config} */
export default {
  content: [
    "./index.html",
    "./src/**/*.{js,ts,jsx,tsx}",
  ],
  theme: {
    extend: {
      colors: {
        "surface-container": "#1f1f28",
        "primary-fixed-dim": "#ddb8ff",
        "tertiary-fixed-dim": "#ddb7ff",
        "surface-container-lowest": "#0d0d16",
        "surface-container-high": "#292932",
        "inverse-on-surface": "#302f39",
        "secondary": "#4cd7f6",
        "surface-container-highest": "#34343d",
        "on-surface-variant": "#cfc2d7",
        "surface-container-low": "#1b1b23",
        "tertiary": "#ddb7ff",
        "surface-variant": "#34343d",
        "on-error": "#690005",
        "on-tertiary": "#490080",
        "on-secondary-fixed-variant": "#004e5c",
        "on-secondary-container": "#00424e",
        "on-primary": "#490080",
        "secondary-container": "#03b5d3",
        "inverse-surface": "#e4e1ed",
        "surface": "#13131b",
        "secondary-fixed": "#acedff",
        "on-primary-fixed": "#2c0051",
        "tertiary-container": "#913bdf",
        "surface-bright": "#393842",
        "on-tertiary-container": "#f6e6ff",
        "primary-fixed": "#f0dbff",
        "primary-container": "#9333ea",
        "error-container": "#93000a",
        "on-primary-container": "#f6e6ff",
        "surface-tint": "#ddb8ff",
        "inverse-primary": "#861fdd",
        "on-primary-fixed-variant": "#6800b4",
        "secondary-fixed-dim": "#4cd7f6",
        "on-secondary": "#003640",
        "outline": "#988ca0",
        "on-tertiary-fixed": "#2c0051",
        "surface-dim": "#13131b",
        "primary": "#ddb8ff",
        "outline-variant": "#4d4354",
        "on-tertiary-fixed-variant": "#6900b3",
        "error": "#ffb4ab",
        "on-background": "#e4e1ed",
        "on-error-container": "#ffdad6",
        "background": "#13131b",
        "on-surface": "#e4e1ed",
        "tertiary-fixed": "#f0dbff",
        "on-secondary-fixed": "#001f26"
      },
      borderRadius: {
        "DEFAULT": "0.25rem",
        "lg": "0.5rem",
        "xl": "0.75rem",
        "2xl": "1.5rem",
        "3xl": "2.5rem",
        "full": "9999px"
      },
      spacing: {
        "gutter": "24px",
        "element-gap": "16px",
        "container-padding-mobile": "20px",
        "container-padding-desktop": "40px",
        "unit": "8px"
      },
      fontFamily: {
        "headline-lg": ["Sora", "sans-serif"],
        "body-md": ["Inter", "sans-serif"],
        "body-lg": ["Inter", "sans-serif"],
        "label-caps": ["Geist", "sans-serif"],
        "headline-md": ["Sora", "sans-serif"],
        "display-xl": ["Sora", "sans-serif"],
        "serif-human": ["Playfair Display", "serif"]
      }
    },
  },
  plugins: [],
}


### Premium CSS & Glassmorphism styling (`frontend/src/index.css`):

In [ ]:
/* frontend/src/index.css */
@import url('https://fonts.googleapis.com/css2?family=Sora:wght@400;500;600;700&family=Inter:wght@300;400;500;600&family=Geist:wght@400;600&family=Playfair+Display:ital,wght@1,500&display=swap');
@import url('https://fonts.googleapis.com/css2?family=Material+Symbols+Outlined:wght,FILL@100..700,0..1&display=swap');

@import "tailwindcss";

@layer base {
  body {
    background-color: #13131b;
    color: #e4e1ed;
    overflow-x: hidden;
    font-family: 'Inter', sans-serif;
    min-height: 100vh;
  }
}

.material-symbols-outlined {
  font-variation-settings: 'FILL' 0, 'wght' 300, 'GRAD' 0, 'opsz' 24;
}

.glass-panel {
  background: rgba(31, 31, 40, 0.4);
  backdrop-filter: blur(24px);
  border: 1px solid rgba(255, 255, 255, 0.08);
  box-shadow: 0 4px 30px rgba(0, 0, 0, 0.15);
  transition: transform 0.5s cubic-bezier(0.175, 0.885, 0.32, 1.275), 
              border-color 0.3s ease, 
              box-shadow 0.3s ease;
}

.glass-panel:hover {
  border-color: rgba(221, 184, 255, 0.2);
  box-shadow: 0 8px 32px rgba(147, 51, 234, 0.1);
}

.liquid-pill-indicator {
  position: absolute;
  left: 8px;
  right: 8px;
  height: 44px;
  background: rgba(147, 51, 234, 0.2);
  border: 1px solid rgba(147, 51, 234, 0.3);
  border-radius: 12px;
  z-index: 0;
  pointer-events: none;
}

.ambilight-shadow {
  box-shadow: 0 0 80px -10px rgba(147, 51, 234, 0.4);
}

.grid-lines {
  background-image: linear-gradient(rgba(255, 255, 255, 0.02) 1px, transparent 1px);
  background-size: 100% 32px;
}

.wave-pulse {
  animation: wave 2s infinite ease-in-out;
}

@keyframes wave {
  0%, 100% { transform: scaleY(1); }
  50% { transform: scaleY(1.4); }
}

.active-upload-glow {
  box-shadow: 0 0 15px rgba(76, 215, 246, 0.5);
  animation: pulse-glow 2s infinite;
}

@keyframes pulse-glow {
  0%, 100% { opacity: 0.8; transform: scale(1); }
  50% { opacity: 1; transform: scale(1.03); }
}

.skeleton-shimmer {
  background: linear-gradient(90deg, 
      rgba(255, 255, 255, 0.03) 25%, 
      rgba(255, 255, 255, 0.08) 50%, 
      rgba(255, 255, 255, 0.03) 75%);
  background-size: 200% 100%;
  animation: shimmer 2.5s infinite linear;
}

@keyframes shimmer {
  0% { background-position: 200% 0; }
  100% { background-position: -200% 0; }
}

.skeleton-block {
  background-color: rgba(255, 255, 255, 0.05);
  border-radius: 0.5rem;
}

.ultra-thin-border {
  border: 0.5px solid rgba(255, 255, 255, 0.08);
}

/* Custom Scrollbar for notes */
::-webkit-scrollbar {
  width: 6px;
  height: 6px;
}

::-webkit-scrollbar-track {
  background: rgba(255, 255, 255, 0.01);
  border-radius: 10px;
}

::-webkit-scrollbar-thumb {
  background: rgba(255, 255, 255, 0.1);
  border-radius: 10px;
}

::-webkit-scrollbar-thumb:hover {
  background: rgba(147, 51, 234, 0.3);
}


### Main UI Dashboard Implementation (`frontend/src/App.jsx`):
Isme Canvas WebGL Background Shader, Drag & Drop uploads, language switcher, Copy/Download functions, aur dynamic GSAP animations implementation shamil hai:

In [ ]:
// frontend/src/App.jsx
import React, { useState, useEffect, useRef } from 'react';
import { gsap } from 'gsap';
import { 
  Video, 
  UploadCloud, 
  Copy, 
  Download, 
  FileText, 
  Search, 
  Bell, 
  Settings, 
  Plus, 
  Play, 
  CheckCircle2, 
  Clock, 
  History, 
  Trash2, 
  ExternalLink 
} from 'lucide-react';

// WebGL Background Shader Component
const ShaderBackground = () => {
  const canvasRef = useRef(null);

  useEffect(() => {
    const canvas = canvasRef.current;
    if (!canvas) return;

    const syncSize = () => {
      const w = window.innerWidth;
      const h = window.innerHeight;
      if (canvas.width !== w || canvas.height !== h) {
        canvas.width = w;
        canvas.height = h;
      }
    };

    window.addEventListener('resize', syncSize);
    syncSize();

    const gl = canvas.getContext('webgl') || canvas.getContext('experimental-webgl');
    if (!gl) return;

    const vs = `
      attribute vec2 a_position;
      varying vec2 v_texCoord;
      void main() {
        v_texCoord = a_position * 0.5 + 0.5;
        gl_Position = vec4(a_position, 0.0, 1.0);
      }
    `;

    const fs = `
      precision highp float;
      varying vec2 v_texCoord;
      uniform float u_time;
      uniform vec2 u_resolution;
      uniform vec2 u_mouse;

      void main() {
          vec2 uv = v_texCoord;
          vec2 center = vec2(0.5, 0.5);
          
          // Create organic movement using sine waves
          float d = distance(uv, center);
          float pulse = sin(d * 8.0 - u_time * 1.5) * 0.08;
          
          // Base Obsidian color
          vec3 obsidian = vec3(0.04, 0.04, 0.08); // #080810 approx
          
          // Amethyst glow
          vec3 amethyst = vec3(0.57, 0.2, 0.91); // #9333EA
          float glow = smoothstep(0.55 + pulse, 0.0, d) * 0.25;
          
          // Aqua hints based on mouse
          vec2 mouseNorm = u_mouse / u_resolution;
          float dMouse = distance(uv, mouseNorm);
          vec3 aqua = vec3(0.02, 0.71, 0.83); // #06B6D4
          float sparkle = pow(max(0.0, sin(uv.x * 15.0 + u_time) * cos(uv.y * 12.0 - u_time)), 40.0);
          float mouseGlow = smoothstep(0.3, 0.0, dMouse) * 0.15;
          
          vec3 finalColor = mix(obsidian, amethyst, glow);
          finalColor += aqua * (sparkle * 0.15 + mouseGlow);
          
          gl_FragColor = vec4(finalColor, 1.0);
      }
    `;

    const cs = (type, src) => {
      const s = gl.createShader(type);
      gl.shaderSource(s, src);
      gl.compileShader(s);
      if (!gl.getShaderParameter(s, gl.COMPILE_STATUS)) {
        console.error(gl.getShaderInfoLog(s));
      }
      return s;
    };

    const prog = gl.createProgram();
    gl.attachShader(prog, cs(gl.VERTEX_SHADER, vs));
    gl.attachShader(prog, cs(gl.FRAGMENT_SHADER, fs));
    gl.linkProgram(prog);
    gl.useProgram(prog);

    const buf = gl.createBuffer();
    gl.bindBuffer(gl.ARRAY_BUFFER, buf);
    gl.bufferData(gl.ARRAY_BUFFER, new Float32Array([-1, -1, 1, -1, -1, 1, 1, 1]), gl.STATIC_DRAW);

    const pos = gl.getAttribLocation(prog, 'a_position');
    gl.enableVertexAttribArray(pos);
    gl.vertexAttribPointer(pos, 2, gl.FLOAT, false, 0, 0);

    const uTime = gl.getUniformLocation(prog, 'u_time');
    const uRes = gl.getUniformLocation(prog, 'u_resolution');
    const uMouse = gl.getUniformLocation(prog, 'u_mouse');

    let mouse = { x: window.innerWidth / 2, y: window.innerHeight / 2 };
    const handleMouseMove = (e) => {
      mouse.x = e.clientX;
      mouse.y = window.innerHeight - e.clientY;
    };
    window.addEventListener('mousemove', handleMouseMove);

    let animationFrameId;
    const render = (t) => {
      syncSize();
      gl.viewport(0, 0, canvas.width, canvas.height);
      if (uTime) gl.uniform1f(uTime, t * 0.001);
      if (uRes) gl.uniform2f(uRes, canvas.width, canvas.height);
      if (uMouse) gl.uniform2f(uMouse, mouse.x, mouse.y);
      gl.drawArrays(gl.TRIANGLE_STRIP, 0, 4);
      animationFrameId = requestAnimationFrame(render);
    };
    render(0);

    return () => {
      window.removeEventListener('resize', syncSize);
      window.removeEventListener('mousemove', handleMouseMove);
      cancelAnimationFrame(animationFrameId);
    };
  }, []);

  return (
    <div className="fixed inset-0 w-full h-full z-0 opacity-40 pointer-events-none">
      <canvas ref={canvasRef} style={{ display: 'block', width: '100%', height: '100%' }} />
    </div>
  );
};

export default function App() {
  const [videoFile, setVideoFile] = useState(null);
  const [videoUrl, setVideoUrl] = useState(null);
  const [isUploading, setIsUploading] = useState(false);
  const [uploadProgress, setUploadProgress] = useState(0);
  const [isProcessing, setIsProcessing] = useState(false);
  const [processingStatus, setProcessingStatus] = useState("Ingesting video frames...");
  const [results, setResults] = useState(null);
  const [selectedLanguage, setSelectedLanguage] = useState("english");
  const [history, setHistory] = useState([]);
  const [sidebarIndex, setSidebarIndex] = useState(0);
  const [errorMessage, setErrorMessage] = useState("");
  const [toastMessage, setToastMessage] = useState("");

  const fileInputRef = useRef(null);
  const videoPlayerRef = useRef(null);
  const navIndicatorRef = useRef(null);
  const langPillRef = useRef(null);
  const resultsContainerRef = useRef(null);
  const sidebarContainerRef = useRef(null);

  // Status message loops
  const statuses = [
    'Ingesting video frames...',
    'Uploading content to Gemini Cloud...',
    'Listening to audio transcript...',
    'Analyzing visual patterns...',
    'Translating key terms to Hinglish...',
    'Crafting summary highlights...',
    'Finalizing neural annotations...'
  ];

  // Load history on mount
  useEffect(() => {
    const saved = localStorage.getItem('vidnotes_history');
    if (saved) {
      try {
        setHistory(JSON.parse(saved));
      } catch (e) {
        console.error(e);
      }
    }
  }, []);

  // Save history helper
  const saveToHistory = (newEntry) => {
    const updated = [newEntry, ...history.filter(h => h.id !== newEntry.id)].slice(0, 10);
    setHistory(updated);
    localStorage.setItem('vidnotes_history', JSON.stringify(updated));
  };

  const deleteHistoryItem = (id, e) => {
    e.stopPropagation();
    const updated = history.filter(h => h.id !== id);
    setHistory(updated);
    localStorage.setItem('vidnotes_history', JSON.stringify(updated));
    if (results && results.id === id) {
      setResults(null);
      setVideoUrl(null);
      setVideoFile(null);
    }
    showToast("History item deleted");
  };

  // Toast notifier helper
  const showToast = (msg) => {
    setToastMessage(msg);
    setTimeout(() => setToastMessage(""), 3000);
  };

  // Sidebar navigation indicator slide
  useEffect(() => {
    const navContainer = sidebarContainerRef.current;
    if (!navContainer || !navIndicatorRef.current) return;
    const items = navContainer.querySelectorAll('.nav-item');
    const target = items[sidebarIndex];
    if (target) {
      const rect = target.getBoundingClientRect();
      const parentRect = navContainer.getBoundingClientRect();
      const top = rect.top - parentRect.top;
      
      gsap.to(navIndicatorRef.current, {
        y: top,
        duration: 0.5,
        ease: "elastic.out(1, 0.75)"
      });
    }
  }, [sidebarIndex]);

  // Language pill slide
  useEffect(() => {
    if (!langPillRef.current) return;
    const btnWidth = langPillRef.current.parentElement.clientWidth / 2;
    const targetX = selectedLanguage === 'hinglish' ? btnWidth - 6 : 0;
    
    gsap.to(langPillRef.current, {
      x: targetX,
      duration: 0.4,
      ease: "power4.out"
    });
  }, [selectedLanguage]);

  // 3D Card tilt effect setup
  useEffect(() => {
    const handleMouseMove = (e, card) => {
      const rect = card.getBoundingClientRect();
      const x = e.clientX - rect.left;
      const y = e.clientY - rect.top;
      const centerX = rect.width / 2;
      const centerY = rect.height / 2;
      const rotateX = (y - centerY) / 20;
      const rotateY = (centerX - x) / 20;
      
      gsap.to(card, {
        rotateX: rotateX,
        rotateY: rotateY,
        duration: 0.5,
        ease: "power2.out"
      });
    };

    const handleMouseLeave = (card) => {
      gsap.to(card, {
        rotateX: 0,
        rotateY: 0,
        duration: 0.8,
        ease: "elastic.out(1, 0.3)"
      });
    };

    const cards = document.querySelectorAll('.glass-panel-3d');
    cards.forEach(card => {
      const onMove = (e) => handleMouseMove(e, card);
      const onLeave = () => handleMouseLeave(card);
      card.addEventListener('mousemove', onMove);
      card.addEventListener('mouseleave', onLeave);
      card._clean = () => {
        card.removeEventListener('mousemove', onMove);
        card.removeEventListener('mouseleave', onLeave);
      };
    });

    return () => {
      cards.forEach(card => {
        if (card._clean) card._clean();
      });
    };
  }, [results, isProcessing]);

  // Stagger entry animation for results
  useEffect(() => {
    if (results && resultsContainerRef.current) {
      gsap.fromTo(resultsContainerRef.current.children, 
        { opacity: 0, y: 30 },
        { 
          opacity: 1, 
          y: 0, 
          duration: 0.8, 
          stagger: 0.15, 
          ease: "power4.out",
          clearProps: "all"
        }
      );
    }
  }, [results]);

  // Dynamic status cycler when processing
  useEffect(() => {
    if (!isProcessing) return;
    let index = 0;
    const interval = setInterval(() => {
      index = (index + 1) % statuses.length;
      setProcessingStatus(statuses[index]);
    }, 4500);

    return () => clearInterval(interval);
  }, [isProcessing]);

  // Handle Drag & Drop events
  const handleDragOver = (e) => {
    e.preventDefault();
  };

  const handleDrop = (e) => {
    e.preventDefault();
    const file = e.dataTransfer.files[0];
    if (file && file.type.startsWith('video/')) {
      processVideoFile(file);
    } else {
      setErrorMessage("Please drop a valid video file (mp4, webm, avi, mov).");
    }
  };

  const selectFile = (e) => {
    const file = e.target.files[0];
    if (file) {
      processVideoFile(file);
    }
  };

  // Main Upload & Processing Flow
  const processVideoFile = async (file) => {
    setErrorMessage("");
    setVideoFile(file);
    setVideoUrl(URL.createObjectURL(file));
    setResults(null);
    setIsUploading(true);
    setUploadProgress(10);

    // Dynamic fake upload progress
    const progressInterval = setInterval(() => {
      setUploadProgress(prev => {
        if (prev >= 90) {
          clearInterval(progressInterval);
          return 90;
        }
        return prev + 10;
      });
    }, 500);

    const formData = new FormData();
    formData.append('video', file);

    try {
      const apiBaseUrl = import.meta.env.VITE_API_URL || 'http://localhost:3000';
      const response = await fetch(`${apiBaseUrl}/api/summarize`, {
        method: 'POST',
        body: formData
      });

      clearInterval(progressInterval);
      setUploadProgress(100);
      setIsUploading(false);
      setIsProcessing(true);
      setProcessingStatus("Gemini is analyzing video transcript...");

      const data = await response.json();

      if (!response.ok) {
        throw new Error(data.error || 'Server error occurred during processing.');
      }

      setIsProcessing(false);
      const newResult = {
        id: Date.now().toString(),
        videoName: file.name,
        processedAt: new Date().toLocaleString(),
        duration: "Computed",
        modelUsed: data.modelUsed || "Gemini 2.5 Flash",
        results: data.results
      };

      setResults(newResult);
      saveToHistory(newResult);
      showToast("Video summarized successfully!");

    } catch (err) {
      clearInterval(progressInterval);
      setIsUploading(false);
      setIsProcessing(false);
      setErrorMessage(err.message || "Failed to analyze video. Make sure server is running and API key is set.");
      console.error(err);
    }
  };

  // Load selected video from history
  const loadHistoryItem = (item) => {
    setErrorMessage("");
    setResults(item);
    setVideoFile({ name: item.videoName });
    setVideoUrl(null); // Local object URL expired, we just show placeholder player or empty state
    showToast(`Loaded notes for: ${item.videoName}`);
  };

  // Copy elements to clipboard
  const copyToClipboard = () => {
    if (!results) return;
    const data = results.results[selectedLanguage];
    const text = `
VidNotes Summary: ${results.videoName}

Executive Summary:
${data.summary}

Key Highlights:
${data.keyPoints.map(p => `- ${p}`).join('\n')}

Detailed Annotated Notes:
${data.notes.map(n => `- ${n}`).join('\n')}
    `;
    navigator.clipboard.writeText(text);
    showToast("Notes copied to clipboard!");
  };

  // Download Markdown file
  const downloadMarkdown = () => {
    if (!results) return;
    const data = results.results[selectedLanguage];
    const content = `
# VidNotes AI | ${results.videoName}
*Processed at: ${results.processedAt}*

## Executive Summary
${data.summary}

## Key Highlights
${data.keyPoints.map(p => `* ${p}`).join('\n')}

## Detailed Study Notes
${data.notes.map(n => `* ${n}`).join('\n')}
    `;
    const blob = new Blob([content], { type: 'text/markdown' });
    const url = URL.createObjectURL(blob);
    const a = document.createElement('a');
    a.href = url;
    a.download = `${results.videoName.replace(/\.[^/.]+$/, "")}_notes_${selectedLanguage}.md`;
    document.body.appendChild(a);
    a.click();
    document.body.removeChild(a);
    showToast("Markdown exported!");
  };

  // Reset workspace
  const handleNewNoteClick = () => {
    setVideoFile(null);
    setVideoUrl(null);
    setResults(null);
    setErrorMessage("");
  };

  return (
    <div className="min-h-screen relative overflow-x-hidden font-body-md text-on-surface select-none">
      <ShaderBackground />
      
      {/* Toast Notification */}
      {toastMessage && (
        <div className="fixed top-6 right-6 z-50 glass-panel px-6 py-3 rounded-full border border-primary/30 flex items-center gap-3 animate-fade-in shadow-[0_0_20px_rgba(147,51,234,0.3)]">
          <div className="w-2 h-2 rounded-full bg-secondary animate-ping"></div>
          <span className="text-sm font-semibold tracking-wider font-label-caps text-on-primary-container">{toastMessage}</span>
        </div>
      )}

      {/* Top App Bar */}
      <header className="fixed top-0 w-full z-50 flex items-center justify-between px-gutter py-4 bg-surface/40 backdrop-blur-3xl border-b border-white/5 shadow-sm">
        <div className="flex items-center gap-4">
          <span className="font-headline-md text-2xl font-bold tracking-tight text-primary cursor-pointer" onClick={handleNewNoteClick}>
            VidNotes
          </span>
          <div className="hidden md:flex bg-white/5 rounded-full px-4 py-1.5 border border-white/5 items-center gap-2">
            <Search className="text-primary w-4 h-4" />
            <input 
              className="bg-transparent border-none outline-none text-body-md text-on-surface-variant w-48 focus:ring-0 placeholder:text-white/20" 
              placeholder="Search library..." 
              type="text"
            />
          </div>
        </div>
      </header>

      {/* Navigation & Sidebar */}
      <nav className="fixed left-6 top-28 bottom-6 w-64 glass-panel rounded-3xl z-40 p-4 flex flex-col justify-between border border-white/10">
        <div className="relative">
          {/* GSAP Indicator background */}
          <div ref={navIndicatorRef} className="liquid-pill-indicator"></div>
          
          <div ref={sidebarContainerRef} className="flex flex-col gap-2 relative">
            <button 
              className={`nav-item flex items-center gap-4 px-4 py-3 rounded-xl transition-all duration-300 w-full text-left z-10 ${sidebarIndex === 0 ? 'text-on-primary-container font-bold' : 'text-on-surface-variant hover:text-on-surface'}`}
              onClick={() => setSidebarIndex(0)}
            >
              <History className="w-5 h-5" />
              <span className="font-label-caps text-[11px] tracking-wider uppercase">Dashboard</span>
            </button>
            <button 
              className={`nav-item flex items-center gap-4 px-4 py-3 rounded-xl transition-all duration-300 w-full text-left z-10 ${sidebarIndex === 1 ? 'text-on-primary-container font-bold' : 'text-on-surface-variant hover:text-on-surface'}`}
              onClick={() => { setSidebarIndex(1); showToast("Library opened"); }}
            >
              <Video className="w-5 h-5" />
              <span className="font-label-caps text-[11px] tracking-wider uppercase">Library</span>
            </button>
          </div>
          
          <div className="mt-8 px-2">
            <button 
              onClick={handleNewNoteClick}
              className="w-full py-4 bg-primary-container text-on-primary-container rounded-2xl font-semibold flex items-center justify-center gap-2 shadow-[0_0_20px_rgba(147,51,234,0.3)] hover:scale-[1.02] active:scale-95 transition-all"
            >
              <Plus className="w-5 h-5" />
              New Note
            </button>
          </div>
        </div>

        {/* History / Recent Panel */}
        <div className="flex flex-col gap-3 mt-4 overflow-y-auto max-h-[300px] pr-1 border-t border-white/5 pt-4">
          <span className="text-[10px] font-label-caps tracking-[0.2em] text-on-surface-variant px-2">RECENT HISTORY</span>
          {history.length === 0 ? (
            <span className="text-xs text-white/20 italic px-2">No past summaries</span>
          ) : (
            history.map((item) => (
              <div 
                key={item.id} 
                onClick={() => loadHistoryItem(item)}
                className={`group flex items-center justify-between p-2 rounded-xl border border-transparent hover:border-white/5 hover:bg-white/5 cursor-pointer transition-all ${results?.id === item.id ? 'bg-primary/10 border-primary/20' : ''}`}
              >
                <div className="flex items-center gap-2 overflow-hidden">
                  <div className="w-8 h-8 rounded-lg bg-white/5 flex items-center justify-center flex-shrink-0">
                    <Video className="w-4 h-4 text-primary" />
                  </div>
                  <div className="flex flex-col overflow-hidden">
                    <span className="text-xs font-semibold text-on-surface truncate pr-2 w-32">{item.videoName}</span>
                    <span className="text-[9px] text-white/30 truncate">{item.processedAt}</span>
                  </div>
                </div>
                <button 
                  onClick={(e) => deleteHistoryItem(item.id, e)}
                  className="opacity-0 group-hover:opacity-100 p-1 hover:text-red-400 transition-opacity"
                >
                  <Trash2 className="w-3.5 h-3.5" />
                </button>
              </div>
            ))
          )}
        </div>


      </nav>

      {/* Main Canvas Area */}
      <main className="ml-80 mr-gutter pt-28 pb-10 flex gap-8">
        
        {/* Left Side: Video & Upload Card */}
        <div className="flex-1 flex flex-col gap-8 min-w-[400px]">
          
          {/* Error Message */}
          {errorMessage && (
            <div className="bg-red-950/40 border border-red-500/30 text-red-200 px-6 py-4 rounded-3xl text-sm">
              {errorMessage}
            </div>
          )}

          <div className="relative glass-panel rounded-[2.5rem] p-4 ambilight-shadow overflow-hidden group">
            
            {/* If no video has been uploaded yet */}
            {!videoFile && !isUploading && !isProcessing && (
              <div 
                onDragOver={handleDragOver}
                onDrop={handleDrop}
                onClick={() => fileInputRef.current.click()}
                className="aspect-video bg-black/40 rounded-[2rem] border-2 border-dashed border-white/10 hover:border-primary/40 flex flex-col items-center justify-center gap-4 cursor-pointer transition-all duration-300"
              >
                <div className="w-16 h-16 rounded-full bg-white/5 flex items-center justify-center border border-white/5 group-hover:scale-105 transition-transform">
                  <UploadCloud className="w-8 h-8 text-primary" />
                </div>
                <div className="text-center">
                  <p className="text-sm font-semibold text-on-surface">Drag & drop your video here</p>
                  <p className="text-xs text-on-surface-variant mt-1">Or click to browse from local computer</p>
                  <p className="text-[10px] text-white/20 mt-4 uppercase tracking-widest font-label-caps">MP4, WEBM, AVI, MOV up to 200MB</p>
                </div>
                <input 
                  type="file" 
                  ref={fileInputRef} 
                  onChange={selectFile} 
                  className="hidden" 
                  accept="video/*" 
                />
              </div>
            )}

            {/* If Uploading to Backend */}
            {isUploading && (
              <div className="aspect-video bg-black/60 rounded-[2rem] flex flex-col items-center justify-center gap-6 relative">
                <div className="text-center z-10">
                  <p className="text-sm font-semibold text-on-surface">Uploading video to server...</p>
                  <p className="text-[11px] text-on-surface-variant mt-1">{uploadProgress}% Uploaded</p>
                </div>
                <div className="w-48 h-1.5 bg-white/5 rounded-full overflow-hidden border border-white/5 z-10">
                  <div className="h-full bg-gradient-to-r from-primary to-secondary transition-all duration-300" style={{ width: `${uploadProgress}%` }}></div>
                </div>
                {/* Visual Glow */}
                <div className="absolute inset-0 bg-primary/10 rounded-[2rem] filter blur-xl opacity-30"></div>
              </div>
            )}

            {/* If Video Ingested on Gemini API */}
            {isProcessing && (
              <div className="aspect-video bg-black/60 rounded-[2rem] flex flex-col items-center justify-center gap-8 relative overflow-hidden">
                {/* Orbital Loader SVG */}
                <div className="relative w-24 h-24 z-10">
                  <svg className="w-full h-full animate-[spin_8s_linear_infinite]" viewBox="0 0 100 100">
                    <circle cx="50" cy="20" fill="#ddb8ff" r="6" className="animate-ping" style={{ animationDuration: '3s' }}></circle>
                    <circle cx="80" cy="50" fill="#4cd7f6" r="6" className="animate-ping" style={{ animationDuration: '2.5s' }}></circle>
                    <circle cx="20" cy="50" fill="#9333ea" r="6" className="animate-ping" style={{ animationDuration: '2s' }}></circle>
                  </svg>
                  <div className="absolute inset-0 blur-xl opacity-30 bg-primary rounded-full"></div>
                </div>
                <div className="flex flex-col items-center gap-2 z-10">
                  <span className="text-on-surface font-headline-md text-xl tracking-wide opacity-90">{processingStatus}</span>
                  <div className="flex gap-1.5 mt-2">
                    <div className="w-1.5 h-1.5 rounded-full bg-primary/40 animate-bounce" style={{ animationDelay: '0.1s' }}></div>
                    <div className="w-1.5 h-1.5 rounded-full bg-primary/40 animate-bounce" style={{ animationDelay: '0.2s' }}></div>
                    <div className="w-1.5 h-1.5 rounded-full bg-primary/40 animate-bounce" style={{ animationDelay: '0.3s' }}></div>
                  </div>
                </div>
                {/* Ingestion progress tracking */}
                <div className="absolute bottom-6 right-6 flex items-center gap-2 px-4 py-2 bg-secondary/20 backdrop-blur-2xl border border-secondary/40 rounded-full">
                  <div className="w-2 h-2 rounded-full bg-secondary animate-pulse"></div>
                  <span className="font-label-caps text-[10px] text-secondary tracking-widest">PROCESSING</span>
                </div>
              </div>
            )}

            {/* Display Video Player once selected (even if no results loaded yet) */}
            {videoFile && !isUploading && !isProcessing && (
              <div className="aspect-video bg-black rounded-[2rem] overflow-hidden relative border border-white/5">
                {videoUrl ? (
                  <video 
                    ref={videoPlayerRef} 
                    src={videoUrl} 
                    className="w-full h-full object-cover" 
                    controls 
                  />
                ) : (
                  <div className="w-full h-full flex flex-col items-center justify-center bg-white/5 relative">
                    <Video className="w-16 h-16 text-primary/40" />
                    <span className="text-xs text-on-surface-variant mt-2 font-semibold italic">Loaded from History: {videoFile.name}</span>
                    <button 
                      onClick={() => fileInputRef.current?.click()} 
                      className="mt-4 px-4 py-2 bg-white/5 hover:bg-white/10 border border-white/10 rounded-full text-xs font-semibold"
                    >
                      Re-upload local file to play
                    </button>
                    <input 
                      type="file" 
                      ref={fileInputRef} 
                      onChange={selectFile} 
                      className="hidden" 
                      accept="video/*" 
                    />
                  </div>
                )}
                
                {/* Active state badge */}
                {results && (
                  <div className="absolute bottom-6 right-6 flex items-center gap-2 px-4 py-2 bg-secondary/20 backdrop-blur-2xl border border-secondary/40 rounded-full active-upload-glow">
                    <div className="w-2 h-2 rounded-full bg-secondary"></div>
                    <span className="font-label-caps text-[10px] text-secondary tracking-widest">SYSTEM ACTIVE</span>
                  </div>
                )}
              </div>
            )}
            
            {/* Custom progress details */}
            {videoFile && !isUploading && !isProcessing && (
              <div className="mt-6 px-4">
                <div className="flex justify-between items-center text-[11px] font-label-caps text-on-surface-variant">
                  <span>File: {videoFile.name}</span>
                  <span>{results ? "Analyzed" : "File Ready"}</span>
                </div>
              </div>
            )}
          </div>

          {/* Bento Details Row */}
          {results && (
            <div className="grid grid-cols-2 gap-6 animate-fade-in">
              <div className="glass-panel p-6 rounded-3xl border border-white/5 flex flex-col gap-4">
                <h3 className="font-headline-md text-md text-primary-fixed-dim">Contextual Insights</h3>
                <p className="text-xs text-on-surface-variant leading-relaxed">
                  Key details and neural concepts have been annotated in the sidebar notes panel. Use tabs to view Hinglish definitions.
                </p>
              </div>
              <div className="glass-panel p-6 rounded-3xl border border-white/5 flex items-center justify-between">
                <div>
                  <h4 className="font-label-caps text-[10px] text-on-surface-variant mb-1">DASHBOARD INTEGRITY</h4>
                  <div className="flex items-center gap-1 text-secondary font-semibold text-xs mt-2">
                    <CheckCircle2 className="w-4 h-4" />
                    <span>{results.modelUsed ? results.modelUsed.replace('gemini-', 'Gemini ').replace('-lite', ' Lite') + ' Active' : 'Gemini 2.5 Active'}</span>
                  </div>
                </div>
              </div>
            </div>
          )}
        </div>

        {/* Right Side: Tabbed Summaries & Annotated Notes */}
        <aside className="w-[440px] flex flex-col gap-6">
          
          {/* Language Switcher Toggle */}
          <div className="glass-panel p-1.5 rounded-full border border-white/10 bg-black/20 flex relative">
            <button 
              onClick={() => setSelectedLanguage("english")}
              className={`flex-1 py-2 text-center text-[11px] font-label-caps z-10 transition-colors uppercase font-bold ${selectedLanguage === 'english' ? 'text-on-primary-container' : 'text-on-surface-variant'}`}
            >
              ENGLISH
            </button>
            <button 
              onClick={() => setSelectedLanguage("hinglish")}
              className={`flex-1 py-2 text-center text-[11px] font-label-caps z-10 transition-colors uppercase font-bold ${selectedLanguage === 'hinglish' ? 'text-on-primary-container' : 'text-on-surface-variant'}`}
            >
              HINGLISH
            </button>
            <div ref={langPillRef} className="absolute top-1.5 bottom-1.5 left-1.5 w-[calc(50%-6px)] bg-primary/20 rounded-full border border-primary/30" id="lang-pill"></div>
          </div>

          {/* Skeleton Loaders during upload/process */}
          {(isUploading || isProcessing || !results) ? (
            <div className="glass-panel p-8 rounded-[2.5rem] border-white/5 flex flex-col gap-6 relative overflow-hidden flex-1 min-h-[500px]">
              <div className="absolute inset-0 grid-lines opacity-10 pointer-events-none"></div>
              
              <div className="flex flex-col gap-4">
                {/* Title Skeleton */}
                <div className="skeleton-block h-8 w-2/3 skeleton-shimmer"></div>
                {/* Body Text Skeleton */}
                <div className="space-y-3 mt-4">
                  <div className="skeleton-block h-4 w-full skeleton-shimmer"></div>
                  <div className="skeleton-block h-4 w-full skeleton-shimmer"></div>
                  <div className="skeleton-block h-4 w-4/5 skeleton-shimmer"></div>
                </div>
                {/* Bullets Skeleton */}
                <div className="flex flex-col gap-6 mt-8">
                  <div className="flex gap-4 items-center">
                    <div className="w-6 h-6 rounded-lg skeleton-block skeleton-shimmer"></div>
                    <div className="skeleton-block h-4 w-3/4 skeleton-shimmer"></div>
                  </div>
                  <div className="flex gap-4 items-center">
                    <div className="w-6 h-6 rounded-lg skeleton-block skeleton-shimmer"></div>
                    <div className="skeleton-block h-4 w-2/3 skeleton-shimmer"></div>
                  </div>
                </div>
              </div>
              
              <div className="mt-auto pt-6 border-t border-white/5">
                <span className="text-xs font-semibold text-primary/70 animate-pulse">
                  {isProcessing ? "Analyzing transcript patterns..." : "Upload a video file to begin..."}
                </span>
              </div>
            </div>
          ) : (
            // Results Panel
            <div ref={resultsContainerRef} className="flex-1 flex flex-col gap-6 min-h-[500px]">
              
              {/* Executive Summary Card */}
              <div className="glass-panel glass-panel-3d p-8 rounded-[2.5rem] border-white/5 flex flex-col gap-4">
                <h2 className="font-headline-md text-lg font-bold text-primary">
                  {selectedLanguage === 'english' ? "Executive Summary" : "Mukhya Saransh (Summary)"}
                </h2>
                <p className="font-serif-human italic text-lg text-on-surface-variant leading-relaxed">
                  "{results.results[selectedLanguage].summary}"
                </p>
              </div>

              {/* Key Highlights Card */}
              <div className="glass-panel glass-panel-3d p-8 rounded-[2.5rem] border-white/5 flex flex-col gap-4">
                <h3 className="font-headline-md text-md font-bold text-secondary">
                  {selectedLanguage === 'english' ? "Key Highlights" : "Khaas Baatein (Key Highlights)"}
                </h3>
                <div className="flex flex-col gap-4 mt-2">
                  {results.results[selectedLanguage].keyPoints.map((point, i) => (
                    <div key={i} className="flex items-start gap-4">
                      <div className="w-5 h-5 rounded-lg bg-secondary/10 flex items-center justify-center border border-secondary/20 mt-1 flex-shrink-0">
                        <CheckCircle2 className="w-3.5 h-3.5 text-secondary" />
                      </div>
                      <span className="text-xs text-on-surface-variant leading-relaxed">{point}</span>
                    </div>
                  ))}
                </div>
              </div>

              {/* Annotated Notes Panel */}
              <div className="flex-1 glass-panel glass-panel-3d rounded-[2.5rem] border-white/5 p-8 flex flex-col gap-6 relative overflow-hidden">
                <div className="absolute inset-0 grid-lines opacity-10 pointer-events-none"></div>
                
                <div className="flex justify-between items-center z-10">
                  <h3 className="font-label-caps text-[10px] tracking-[0.2em] text-on-surface-variant">
                    {selectedLanguage === 'english' ? "ANNOTATED STUDY NOTES" : "ANNOTATED PADHAI NOTES"}
                  </h3>
                </div>
                
                <div className="space-y-4 z-10 overflow-y-auto max-h-[220px] pr-1">
                  {results.results[selectedLanguage].notes.map((note, i) => (
                    <div key={i} className="relative pl-6">
                      <div className="absolute left-0 top-2.5 w-1.5 h-1.5 rounded-full bg-primary shadow-[0_0_8px_#ddb8ff]"></div>
                      <p className="text-xs text-on-surface leading-loose">{note}</p>
                    </div>
                  ))}
                </div>
                
                {/* Download / Actions Panel */}
                <div className="mt-auto space-y-3 z-10">
                  <button 
                    onClick={copyToClipboard}
                    className="w-full py-3.5 rounded-full bg-gradient-to-r from-surface-container-high to-surface-container border border-white/10 text-on-surface text-xs font-semibold flex items-center justify-center gap-2 hover:border-white/20 hover:scale-[1.01] transition-all"
                  >
                    <Copy className="w-4 h-4" />
                    Copy Notes
                  </button>
                  <button 
                    onClick={downloadMarkdown}
                    className="w-full py-3.5 rounded-full bg-gradient-to-r from-primary to-primary-container text-on-primary-container text-xs font-bold flex items-center justify-center gap-2 shadow-[0_4px_20px_rgba(147,51,234,0.4)] hover:shadow-[0_0_30px_rgba(147,51,234,0.6)] hover:scale-[1.01] transition-all relative overflow-hidden group"
                  >
                    <div className="absolute inset-0 bg-white/10 opacity-0 group-hover:opacity-100 transition-opacity duration-700 blur-xl"></div>
                    <Download className="w-4 h-4" />
                    Export Markdown Notes
                  </button>
                </div>
              </div>

            </div>
          )}
        </aside>

      </main>
    </div>
  );
}


## 3. Run Instructions (Kaise Run Karein)

Aap is project ko do simple terminals mein local machine par run kar sakte hain:

### Step 1: Running Backend Server
1. Apni Gemini API Key `.env` mein save karein.
2. Terminal open karke `backend` folder mein run karein:
   ```bash
   cd backend
   npm start
   ```

### Step 2: Running React Frontend
1. Dusre terminal window mein `frontend` folder mein run karein:
   ```bash
   cd frontend
   npm run dev
   ```
2. Browser mein standard Vite port (e.g. `http://localhost:5173`) open karke interface check karein!

## 4. Python API Testing Sandbox (Bonus)

Agar aap direct python environment se kisi video ko summarize karna chahte hain, toh official `google-genai` Python library use karke is sandbox code block ko execute kar sakte hain:

In [ ]:
# install package if needed:
# !pip install google-genai

from google import genai
import os
import time

# Initialize client (looks for GEMINI_API_KEY env variable)
api_key = os.environ.get('GEMINI_API_KEY', 'YOUR_API_KEY_HERE')
client = genai.Client(api_key=api_key)

# Put your video filename here
video_path = 'sample_video.mp4'

if os.path.exists(video_path):
    print(f'Uploading {video_path} to Gemini...')
    video_file = client.files.upload(file=video_path)
    
    # Poll file status until it is ACTIVE
    while video_file.state.name == 'PROCESSING':
        print('Ingesting video... waiting 5 seconds')
        time.sleep(5)
        video_file = client.files.get(name=video_file.name)
        
    if video_file.state.name == 'FAILED':
        print('Video ingestion failed!')
    else:
        print('Video processed successfully. Requesting summary...')
        response = client.models.generate_content(
            model='gemini-2.5-flash',
            contents=[
                video_file,
                'Summarize this video and output highlights in both English and Hinglish.'
            ]
        )
        print('\n--- Gemini Output ---')
        print(response.text)
else:
    print(f'Please place a sample video file named "{video_path}" in the folder to test this code.')